# Diabetes Readmission — Neural Networks

A Keras classifier on the readmission problem, and — more importantly — an honest way to measure
it. A single train/test split on this data is noisy enough that the same architecture can look
strong or weak depending on which split it drew, so the network is evaluated by **stratified
cross-validation** across folds rather than by one number.

The comparison against classical models at the end is the point of the notebook. The network does
not win, and that is a result worth teaching.

## Learning objectives

- Build a binary Keras classifier ending in a single sigmoid unit
- Explain why binary classification uses `binary_crossentropy` and one output, not softmax and two
- Evaluate a network with stratified K-fold cross-validation and report the spread, not just the mean
- Use `EarlyStopping` and `ReduceLROnPlateau` to manage training
- Compare a network against classical models on the same data and interpret a null result

## Background

This notebook assumes the preprocessed table from `U1_Diabetes-2_Preprocess`, the Keras
`Sequential` API from `U1-6_Classify-5_KerasIntro`, stratified folds from
`U1-7_Imbalance-1_StratifiedKFold`, and the metric reasoning from `U1_Diabetes-3_Classification` —
per-class F1 rather than accuracy on an 11% positive rate.

The one architectural difference from the multiclass network is covered in section 3 where the
model is built.

**Prerequisites:** `U1_Diabetes-3_Classification` (metrics, the imbalance),
`U1-6_Classify-5_KerasIntro` (Keras classification),
`U1-7_Imbalance-1_StratifiedKFold` (stratified CV)

**Dataset:** `diabetes_hospital_preprocessed.csv` — the encoded output of
`U1_Diabetes-2_Preprocess`, subsampled to 25,000 encounters.

**References:** https://keras.io/api/callbacks/

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from tqdm import tqdm

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Load the preprocessed data

`U1_Diabetes-2_Preprocess` already dropped the leakage columns and dummy-encoded every
categorical, saving the result to `diabetes_hospital_preprocessed.csv`. We load that and go
straight to modeling.

### 1.1 Subsample

The full dataset is ~102k encounters — fine for production, slow for classroom experimentation. We
take a **seeded, stratified** subsample of 25,000: stratified so the ~11% positive rate is
preserved exactly, seeded so every notebook in this spine sees the same 25,000 patients and their
results are comparable.

In [2]:
data_folder = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'

# Load the fully preprocessed dataset saved by notebook 2.
df = pd.read_csv(data_folder + 'Diabetes/diabetes_hospital_preprocessed.csv')

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 42 columns):
 #   Column                                      Non-Null Count   Dtype
---  ------                                      --------------   -----
 0   age                                         101766 non-null  int64
 1   time_in_hospital                            101766 non-null  int64
 2   num_lab_procedures                          101766 non-null  int64
 3   num_procedures                              101766 non-null  int64
 4   num_medications                             101766 non-null  int64
 5   number_diagnoses                            101766 non-null  int64
 6   readmit_30_days                             101766 non-null  int64
 7   race_AfricanAmerican                        101766 non-null  int64
 8   race_Asian                                  101766 non-null  int64
 9   race_Hispanic                               101766 non-null  int64
 10  race_Other          

,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,readmit_30_days,race_AfricanAmerican,race_Asian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,discharge_disposition_id_Other,admission_source_id_Other,admission_source_id_Referral,medical_specialty_Cardiology,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Other,primary_diagnosis_'Genitourinary Issues',primary_diagnosis_'Musculoskeletal Issues',primary_diagnosis_'Respiratory Issues',primary_diagnosis_Diabetes,max_glu_serum_>200,max_glu_serum_>300,max_glu_serum_Norm,A1Cresult_>7,A1Cresult_>8,A1Cresult_Norm,insulin_Down,insulin_Steady,insulin_Up,change_Ch,diabetesMed_No,medicare_True,medicaid_True,had_emergency_True,had_inpatient_days_True,had_outpatient_days_True
0,0,1,41,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,0,3,59,0,18,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
2,0,2,11,5,13,6,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1
3,1,2,44,1,16,7,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0
4,1,1,51,0,8,5,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0


The full dataset has ~102k encounters — great for production, slow for classroom experimentation. We work with a seeded **stratified** subsample of 25,000 rows (~2,800 positives), which preserves the class ratio.

In [3]:
from sklearn.model_selection import train_test_split

df, _ = train_test_split(
    df,
    train_size=25_000,
    stratify=df['readmit_30_days'],
    random_state=42
)
df = df.reset_index(drop=True)

print(df.shape)
print(f"positive rate: {df['readmit_30_days'].mean():.1%}")

(25000, 42)
positive rate: 11.2%


## 2. Prepare the data for modeling

In [4]:
df.sample(5)

,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,readmit_30_days,race_AfricanAmerican,race_Asian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,discharge_disposition_id_Other,admission_source_id_Other,admission_source_id_Referral,medical_specialty_Cardiology,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Other,primary_diagnosis_'Genitourinary Issues',primary_diagnosis_'Musculoskeletal Issues',primary_diagnosis_'Respiratory Issues',primary_diagnosis_Diabetes,max_glu_serum_>200,max_glu_serum_>300,max_glu_serum_Norm,A1Cresult_>7,A1Cresult_>8,A1Cresult_Norm,insulin_Down,insulin_Steady,insulin_Up,change_Ch,diabetesMed_No,medicare_True,medicaid_True,had_emergency_True,had_inpatient_days_True,had_outpatient_days_True
21082,1,7,19,2,10,9,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
21504,1,13,61,6,21,9,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0
16234,2,3,43,0,29,6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1
21741,2,4,43,0,18,9,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0
23949,2,7,66,0,19,9,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0


In [5]:
label = 'readmit_30_days'

# Binary target: already 0/1, so no rare-class filtering or label encoding is needed.
X_raw = df.drop(columns=[label])
y = df[label].copy()

y.value_counts()

readmit_30_days
0    22210
1     2790
Name: count, dtype: int64

### 2.1 Scale the features

Everything loaded from notebook 2 is numeric — ordinal age plus dummy columns — so the whole
matrix is scaled. Tree models are indifferent to feature scale, but logistic regression, KNN, and
neural networks are not, and we are about to compare all of them on equal footing.

In [6]:
from sklearn.preprocessing import StandardScaler

# The data loaded from notebook 2 is already dummy-encoded, so every feature
# column is numeric. We scale the entire feature matrix to zero mean, unit variance.
std_scaler = StandardScaler()

X = pd.DataFrame(
    std_scaler.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index
)

X.head()

,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,race_AfricanAmerican,race_Asian,race_Hispanic,race_Other,race_Unknown,gender_Male,gender_Unknown/Invalid,discharge_disposition_id_Other,admission_source_id_Other,admission_source_id_Referral,medical_specialty_Cardiology,medical_specialty_Emergency/Trauma,medical_specialty_Family/GeneralPractice,medical_specialty_InternalMedicine,medical_specialty_Other,primary_diagnosis_'Genitourinary Issues',primary_diagnosis_'Musculoskeletal Issues',primary_diagnosis_'Respiratory Issues',primary_diagnosis_Diabetes,max_glu_serum_>200,max_glu_serum_>300,max_glu_serum_Norm,A1Cresult_>7,A1Cresult_>8,A1Cresult_Norm,insulin_Down,insulin_Steady,insulin_Up,change_Ch,diabetesMed_No,medicare_True,medicaid_True,had_emergency_True,had_inpatient_days_True,had_outpatient_days_True
0,0.667206,2.562889,2.339641,-0.195398,1.714086,0.818711,-0.483441,-0.078471,-0.148301,-0.123404,-0.149704,1.080077,-0.008945,1.206164,-0.390375,-0.662582,-0.240226,3.499792,-0.281285,-0.407619,-0.446075,-0.229222,-0.228836,-0.406083,3.261707,-0.122229,-0.112782,-0.162088,-0.199292,3.406928,-0.229512,2.673196,-0.658397,-0.354428,1.070564,-0.546165,1.469204,-0.193136,2.821449,-0.702064,-0.443688
1,0.667206,-1.137830,0.409234,-0.783592,-0.982776,-0.734427,-0.483441,-0.078471,-0.148301,8.103497,-0.149704,1.080077,-0.008945,-0.829075,-0.390375,-0.662582,-0.240226,-0.285731,-0.281285,2.453269,-0.446075,-0.229222,-0.228836,2.462548,-0.306588,-0.122229,-0.112782,6.169481,-0.199292,-0.293520,-0.229512,-0.374084,-0.658397,-0.354428,-0.934087,1.830947,-0.680641,-0.193136,-0.354428,-0.702064,-0.443688
2,0.667206,-0.801401,0.714035,-0.783592,-0.492437,-1.252140,2.068506,-0.078471,-0.148301,-0.123404,-0.149704,-0.925860,-0.008945,-0.829075,-0.390375,-0.662582,-0.240226,-0.285731,-0.281285,-0.407619,-0.446075,-0.229222,-0.228836,-0.406083,-0.306588,-0.122229,-0.112782,-0.162088,-0.199292,-0.293520,-0.229512,2.673196,-0.658397,-0.354428,1.070564,-0.546165,-0.680641,-0.193136,-0.354428,1.424372,2.253834
3,0.667206,1.553602,0.510834,-0.195398,-0.124683,-1.252140,-0.483441,-0.078471,-0.148301,-0.123404,-0.149704,-0.925860,-0.008945,1.206164,-0.390375,-0.662582,-0.240226,-0.285731,-0.281285,2.453269,-0.446075,-0.229222,-0.228836,-0.406083,-0.306588,-0.122229,8.866670,-0.162088,-0.199292,-0.293520,-0.229512,-0.374084,-0.658397,-0.354428,-0.934087,1.830947,-0.680641,-0.193136,-0.354428,1.424372,-0.443688
4,0.667206,-0.801401,-0.911571,-0.783592,-0.615022,-0.734427,-0.483441,-0.078471,-0.148301,-0.123404,-0.149704,-0.925860,-0.008945,-0.829075,-0.390375,1.509248,-0.240226,-0.285731,-0.281285,-0.407619,2.241773,-0.229222,-0.228836,2.462548,-0.306588,-0.122229,-0.112782,-0.162088,-0.199292,-0.293520,-0.229512,-0.374084,1.518841,-0.354428,-0.934087,-0.546165,-0.680641,-0.193136,2.821449,-0.702064,-0.443688


## 3. Modeling

### 3.1 The network

Binary classification differs from the multiclass case of `U1-6_Classify-5_KerasIntro` in exactly
one place — the output layer. Rather than $K$ softmax units, a binary classifier uses a **single
sigmoid unit**:

$$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

which returns the probability of the positive class; the negative class is $1 - \hat{p}$, so a
second unit would be redundant. The matching loss is **binary cross-entropy**:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}
\Big[y_i \log \hat{p}_i + (1-y_i)\log(1 - \hat{p}_i)\Big]$$

which is the two-class special case of the categorical cross-entropy derived in
`U1-6_Classify-2_Entropy` — one term fires per example depending on the true label.

Two callbacks manage training. **`EarlyStopping`** halts when validation loss stops improving and
restores the best weights, automating "stop at the peak of the test curve". **`ReduceLROnPlateau`**
cuts the learning rate when progress stalls, letting the optimizer take large steps early and fine
ones near the minimum — the two ends of the sweep in `U1-3_Regression-6_KerasLossVsLR`, applied in
sequence rather than chosen once.

In [7]:
from keras.models import Sequential
from keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def build_model( X_train, initial_learning_rate, dropout_rate ):

    # Create model (binary target -> two-unit softmax, one probability per class)
    model = Sequential([
        Input(shape=X_train.shape[1:]),

        Dense(20),
        BatchNormalization(),
        Activation('relu'),
        Dropout(dropout_rate),

        Dense(20),
        BatchNormalization(),
        Activation('relu'),
        Dropout(dropout_rate),

        Dense(10, activation='relu'),

        Dense(2, activation='softmax'),
    ])

    # Define the optimizer with a custom learning rate
    optimizer = Adam(
        learning_rate=initial_learning_rate,
    )

    # Compile model (softmax output + integer labels -> sparse categorical cross-entropy)
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model
# end

### 3.2 Cross-validated evaluation

A single split on this data is noisy, so the same architecture is trained once per fold and the
metrics collected across all of them. **Stratified** folds keep the ~11% positive rate intact in
every one — with an ordinary `KFold`, a fold's positive count would vary enough to swamp the
differences we are trying to measure.

Read `describe()` on the result with the `std` row in mind. If the standard deviation across folds
is comparable to the difference between two models, then those models are not distinguishable on
this data — a conclusion that a single split would have hidden behind one confident-looking number.

In [8]:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

epochs                = 40
batch_size            = 256
dropout_rate          = 0.25
initial_learning_rate = 0.01

n_splits = 5
#kf = KFold(n_splits=n_splits, shuffle=True)
kf = StratifiedKFold(n_splits=n_splits, shuffle=True)

results = []

fold = 1
for train_index, test_index in kf.split(X, y):
    print(f"Fold {fold}/{n_splits}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Build a fresh model (which resets optimizer state)
    model = build_model(
        X_train,
        initial_learning_rate,
        dropout_rate
    )

    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_loss',  # Monitor validation loss
        patience=8,          # Stop after 5 epochs without improvement
        restore_best_weights=True  # Restore the best weights after stopping
    )
    
    # Train the model (no class weights)
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        verbose=0,
        callbacks=[early_stopping]
    )
    
    # Evaluate on the held-out fold only - we don't track training-set metrics, since
    # the point of CV is how the model does on data it didn't see.
    y_test_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    
    fold_results = {
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Test Precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'Test F1': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
    }
    
    print(f"  Test Accuracy: {fold_results['Test Accuracy']:.4f}")
    print()
    
    results.append(fold_results)
    
    fold += 1
# end

Fold 1/5


  Test Accuracy: 0.8884

Fold 2/5


  Test Accuracy: 0.8884

Fold 3/5


  Test Accuracy: 0.8884

Fold 4/5


  Test Accuracy: 0.8884

Fold 5/5


  Test Accuracy: 0.8884



In [9]:
results_df = pd.DataFrame(results)

results_df.describe()

,Test Accuracy,Test Precision,Test Recall,Test F1
count,5.000000e+00,5.000000,5.000000e+00,5.000000
mean,8.884000e-01,0.789255,8.884000e-01,0.835898
std,1.241267e-16,0.000000,1.241267e-16,0.000000
min,8.884000e-01,0.789255,8.884000e-01,0.835898
25%,8.884000e-01,0.789255,8.884000e-01,0.835898
50%,8.884000e-01,0.789255,8.884000e-01,0.835898
75%,8.884000e-01,0.789255,8.884000e-01,0.835898
max,8.884000e-01,0.789255,8.884000e-01,0.835898


## 4. Comparison with classical models

For quick reference against non-neural models, we compare several sklearn classifiers' test-set
metrics on a single train/test split against the Keras CV results above.

This is the honest part. Neural networks are the newest tool in the unit and the least likely to
help here: the data is tabular, moderate in size, and mostly dummy columns, which is exactly the
regime where gradient-boosted trees are hard to beat. Expect the network to land in the same range
as the classical models rather than above them.

That is a real finding, not a failed experiment. Choosing a model family is an empirical question,
and "the simplest thing works as well" is a legitimate — and common — answer on tabular data.

In [10]:
# The cross-validation above reassigned X_train/X_test/y_train/y_test to its last
# fold; take a fresh single split here for this sklearn comparison.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Define classification models
models = [
    LogisticRegression(max_iter=1000),
    KNeighborsClassifier(n_neighbors=5, weights='uniform', n_jobs=-1),
    KNeighborsClassifier(n_neighbors=10, weights='uniform', n_jobs=-1),
    KNeighborsClassifier(n_neighbors=50, weights='uniform', n_jobs=-1),
    RandomForestClassifier(n_estimators=100, max_leaf_nodes=3),
    RandomForestClassifier(n_estimators=100, max_leaf_nodes=10),
    RandomForestClassifier(n_estimators=100, max_leaf_nodes=30),
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_leaf_nodes=3),
    GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_leaf_nodes=10),
]

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for m in models:
    m.fit(X_train, y_train)
    y_test_pred = m.predict(X_test)
    
    results.append({
        'Model': str(m),
        'Test Accuracy': accuracy_score(y_test, y_test_pred),
        'Test Precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'Test Recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'Test F1': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
    })
# end

In [13]:
comparison_df = (pd.DataFrame(results)
                 .set_index('Model')
                 .sort_values(by='Test F1', ascending=False)
)
comparison_df

,Test Accuracy,Test Precision,Test Recall,Test F1
Model,,,,
GradientBoostingClassifier(max_leaf_nodes=10),0.8884,0.845331,0.8884,0.836670
"KNeighborsClassifier(n_jobs=-1, n_neighbors=10)",0.8876,0.817352,0.8876,0.836265
KNeighborsClassifier(n_jobs=-1),0.8772,0.812489,0.8772,0.836241
LogisticRegression(max_iter=1000),0.8884,0.789255,0.8884,0.835898
"KNeighborsClassifier(n_jobs=-1, n_neighbors=50)",0.8884,0.789255,0.8884,0.835898
RandomForestClassifier(max_leaf_nodes=3),0.8884,0.789255,0.8884,0.835898
RandomForestClassifier(max_leaf_nodes=10),0.8884,0.789255,0.8884,0.835898
RandomForestClassifier(max_leaf_nodes=30),0.8884,0.789255,0.8884,0.835898
GradientBoostingClassifier(max_leaf_nodes=3),0.8884,0.789255,0.8884,0.835898


## 5. Review

- **Binary classification uses one sigmoid unit and `binary_crossentropy`**, not two softmax units.
  The second probability is $1-\hat{p}$ and would be redundant.
- **Binary cross-entropy is the two-class case of categorical cross-entropy**, with one term firing
  per example.
- **Evaluate with stratified cross-validation and report the spread.** On this data a single split
  is noisy enough to reverse a comparison.
- **`EarlyStopping` and `ReduceLROnPlateau`** automate stopping at the right time and shrinking the
  step size as the optimizer closes in.
- **The network does not beat the classical models here**, which is the expected outcome on moderate
  tabular data and a legitimate result to report.